# Data

Synthetic profiles are generated from a model calibrated to the aggregate per-hour statistics of the original (confidential) dataset — the per-hour mean/std of log-traffic and an hour-to-hour correlation structure — with a small set of labeled anomalies injected on top (outages, spikes, phase shifts, inversions). This gives the project something the original never had: **ground-truth labels**, so the detection pipeline below can be evaluated quantitatively (precision/recall/F1/ROC-AUC), not just visually.

The real dataset (`dataset.xlsx`) is then merged in alongside the synthetic profiles. Its site identifiers are replaced with the same `S####` scheme used for the synthetic data (continuing the numbering so IDs never collide) — only the identifier changes, the traffic values themselves are untouched. The real rows have no ground-truth label and are excluded from the quantitative evaluation section for that reason, but they do flow through data loading, normalization, training, scoring, and thresholding like every other row.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

#Aggregate statistics fit from the original (confidential) dataset: per-hour
#mean/std of log-traffic, plus how strongly a site's own overall scale
#("big site" vs "small site") and hour-to-hour shape correlate. Only these
#summary numbers are used here to build the synthetic generator below.
LOG_MEAN = [4.0383, 3.7801, 3.4943, 3.2122, 2.9655, 2.8619, 3.0614, 3.5691,
           3.8667, 3.9988, 4.0922, 4.1426, 4.2595, 4.2990, 4.2634, 4.2586,
           4.2860, 4.3134, 4.3216, 4.2815, 4.2708, 4.3192, 4.3138, 4.2231]
LOG_STD = [0.6692, 0.6844, 0.7133, 0.7258, 0.7344, 0.7325, 0.7215, 0.6559,
          0.6287, 0.6140, 0.6001, 0.5960, 0.5681, 0.5798, 0.5863, 0.5858,
          0.5780, 0.5787, 0.5930, 0.6055, 0.6208, 0.6202, 0.6305, 0.6458]
SITE_SCALE_FRACTION = 0.84  # share of variance from a site's own overall traffic level
HOUR_CORRELATION = 0.84     # hour-to-hour correlation of the residual daily shape

N_NORMAL = 705
ANOMALY_COUNTS = {"outage": 10, "spike": 10, "phase_shift": 10, "inverted": 8}
SEED = 42
REAL_DATA_PATH = "dataset.xlsx"

def _sample_ar1(n, n_hours, rho, rng):
    """Stationary AR(1) noise process across the 24 hours (unit variance)."""
    z = np.zeros((n, n_hours))
    z[:, 0] = rng.normal(0, 1, n)
    sigma_innov = np.sqrt(1 - rho ** 2)
    for h in range(1, n_hours):
        z[:, h] = rho * z[:, h - 1] + rng.normal(0, sigma_innov, n)
    return z

def _make_normal_profiles(n, rng):
    log_mean = np.array(LOG_MEAN)
    log_std = np.array(LOG_STD)
    site_effect = rng.normal(0, 1, n)                        # one draw per site ("big" vs "small")
    shape_noise = _sample_ar1(n, 24, HOUR_CORRELATION, rng)   # correlated hour-to-hour variation
    log_x = (log_mean[None, :]
            + site_effect[:, None] * np.sqrt(SITE_SCALE_FRACTION) * log_std[None, :]
            + shape_noise * np.sqrt(1 - SITE_SCALE_FRACTION) * log_std[None, :])
    return np.exp(log_x)

def _inject_anomaly(profile, kind, rng):
    """Deliberately break the normal daily shape in a labeled, recognizable way."""
    profile = profile.copy()
    if kind == "outage":
        start, dur = rng.integers(0, 20), rng.integers(3, 6)
        profile[start:start + dur] *= rng.uniform(0.02, 0.1)
    elif kind == "spike":
        start, dur = rng.integers(0, 22), rng.integers(1, 3)
        profile[start:start + dur] *= rng.uniform(3.0, 6.0)
    elif kind == "phase_shift":
        profile = np.roll(profile, rng.integers(6, 18))
    elif kind == "inverted":
        m = profile.mean()
        profile = np.clip(2 * m - profile, profile.min() * 0.5, None)
    return profile

def generate_synthetic_dataset(n_normal=N_NORMAL, anomaly_counts=None, seed=SEED):
    rng = np.random.default_rng(seed)
    anomaly_counts = anomaly_counts or ANOMALY_COUNTS
    n_anomalies = sum(anomaly_counts.values())
    base_profiles = _make_normal_profiles(n_normal + n_anomalies, rng)

    rows, labels, kinds = [], [], []
    idx = 0
    for _ in range(n_normal):
        rows.append(base_profiles[idx]); labels.append(0); kinds.append("normal")
        idx += 1
    for kind, count in anomaly_counts.items():
        for _ in range(count):
            rows.append(_inject_anomaly(base_profiles[idx], kind, rng))
            labels.append(1); kinds.append(kind)
            idx += 1

    n_total = len(rows)
    site_ids = np.array([f"S{i:04d}" for i in range(1, n_total + 1)])
    order = rng.permutation(n_total)
    rows = np.clip(np.array(rows)[order], 0.1, None)
    labels = np.array(labels)[order]
    kinds = np.array(kinds)[order]
    site_ids = site_ids[order]

    df = pd.DataFrame(rows, columns=list(range(24)))
    df.insert(0, "site", site_ids)
    ground_truth = pd.DataFrame({"site": site_ids, "is_anomaly": labels, "anomaly_type": kinds})
    return df, ground_truth

def load_relabeled_real_dataset(path, id_offset):
    """Load the real dataset and replace its site identifiers with the same
    S#### scheme used for synthetic sites, continuing the numbering so IDs
    never collide. Only the identifier is replaced - traffic values are
    untouched. These rows carry no ground-truth anomaly label.
    """
    real_df = pd.read_excel(path)
    real_hour_cols = [c for c in real_df.columns if c != "site"]
    real_df = real_df[real_hour_cols].copy()
    real_df.columns = list(range(24))
    n = len(real_df)
    real_df.insert(0, "site", [f"S{i:04d}" for i in range(id_offset, id_offset + n)])
    return real_df

synthetic_df, ground_truth = generate_synthetic_dataset()

if os.path.exists(REAL_DATA_PATH):
    real_df = load_relabeled_real_dataset(REAL_DATA_PATH, id_offset=len(synthetic_df) + 1)
    df = pd.concat([synthetic_df, real_df], ignore_index=True)
    print(f"Merged {len(synthetic_df)} synthetic + {len(real_df)} real (re-labeled) profiles "
         f"= {len(df)} total. Ground-truth labels only exist for the synthetic subset.")
else:
    df = synthetic_df
    print(f"{REAL_DATA_PATH} not found - continuing with the synthetic-only dataset ({len(df)} rows).")

print("Injected anomalies:", int(ground_truth["is_anomaly"].sum()), "/", len(ground_truth),
     f"({ground_truth['is_anomaly'].mean():.1%})")
display(ground_truth["anomaly_type"].value_counts())

missing_values = df.isnull().sum()
print("Total missing values:", int(df.isnull().sum().sum()))

hour_cols = [col for col in df.columns if col != "site"]

traffic_summary = df[hour_cols].describe()
display(traffic_summary)

In [ ]:
print("Number of base stations:", df.shape[0])
print("Number of hourly traffic features:", len(hour_cols))

if len(hour_cols) == 24:
    print("Dataset is suitable for SDA input: each base station has 24 hourly values.")
else:
    print("Warning: hourly columns are not equal to 24. Please check the dataset.")

In [ ]:
site_ids = df["site"].copy()

print("Number of site IDs:", len(site_ids))
display(site_ids.head())

X_raw = df[hour_cols].copy()

print("Shape of X_raw:", X_raw.shape)
display(X_raw.head())

X_raw_np = X_raw.values.astype("float32")

print("Shape of X_raw_np:", X_raw_np.shape)
print("Data type:", X_raw_np.dtype)

plt.figure(figsize=(10, 5))
for i in range(5):
    plt.plot(hour_cols, X_raw_np[i], marker="o", label=site_ids.iloc[i])
plt.xlabel("Hour")
plt.ylabel("Traffic value")
plt.title("Example of Raw 24-Hour Traffic Profiles")
plt.legend()
plt.grid(True)
plt.show()

# Mean Normalization

Each profile is divided by its daily mean so that its average over the 24 hours equals 1. This removes the absolute traffic magnitude (scale) differences between base stations while preserving the relative hourly variations of the daily curve. A value of 1 corresponds to an hour at the daily-average level, values above 1 are above-average hours, and values below 1 are below-average hours.

In [ ]:
daily_mean = X_raw_np.mean(axis=1)
zero_mean_count = np.sum(daily_mean == 0)

safe_mean = np.where(daily_mean == 0, 1.0, daily_mean)
X_norm = X_raw_np / safe_mean[:, np.newaxis]

print(X_norm.shape)

In [ ]:
X_norm_df = pd.DataFrame(X_norm, columns=hour_cols)
X_norm_df.insert(0, "site", site_ids.values)

display(X_norm_df.head())


In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_val = train_test_split(
    X_norm,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print("Shape of X_train:", X_train.shape)
print("Shape of X_val:", X_val.shape)

In [ ]:
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
input_dim = X_train.shape[1]

def build_autoencoder(input_dim, encoder_units, learning_rate=0.001,
                      output_activation="linear"):
    """Dense symmetric autoencoder.
    encoder_units: list, last entry is the bottleneck (latent) dimension.
    """
    input_layer = Input(shape=(input_dim,), name="input_layer")
    x = input_layer

    #Encoder
    for i, units in enumerate(encoder_units):
        x = Dense(units, activation="relu", name=f"encoder_dense_{i+1}")(x)

    #Decoder (mirror, excluding the bottleneck)
    decoder_units = list(reversed(encoder_units[:-1]))
    for i, units in enumerate(decoder_units):
        x = Dense(units, activation="relu", name=f"decoder_dense_{i+1}")(x)

    output_layer = Dense(input_dim, activation=output_activation,
                         name="output_layer")(x)

    model = Model(inputs=input_layer, outputs=output_layer,
                  name="simple_dense_autoencoder")
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss="mse")
    return model

In [ ]:
import itertools
import tensorflow as tf

#Architectures to compare: encoder layer sizes (last entry = bottleneck).
#Shallow (1 encoder layer): that layer IS the bottleneck.
#Deep (2 encoder layers): first layer compresses, second is the bottleneck.
#Bottleneck range 4-8 on a 24-dim input covers the useful compression spectrum.
ARCHITECTURES = {
    "SDA_24_4_24":        [4],
    "SDA_24_6_24":        [6],
    "SDA_24_8_24":        [8],
    "SDA_24_12_4_12_24":  [12, 4],
    "SDA_24_12_6_12_24":  [12, 6],
    "SDA_24_16_8_16_24":  [16, 8],
}
LEARNING_RATES = [1e-3, 5e-4]
SEEDS = [42, 1, 7]          # multiple seeds -> robust, less noisy selection

EPOCHS = 300
BATCH_SIZE = 32
PATIENCE = 20

TUNING_RESULTS_PATH = "tuning_results.csv"
FORCE_RETRAIN_SEARCH = False  # set True to ignore the cache and re-run the search

def arch_text_of(encoder_units):
    decoder_units = list(reversed(encoder_units[:-1]))
    chain = [input_dim] + list(encoder_units) + decoder_units + [input_dim]
    return " -> ".join(map(str, chain))

def sep_ratio_from_model(model, X):
    recon = model.predict(X, verbose=0)
    s = np.sqrt(np.mean((X - recon) ** 2, axis=1))
    return float(np.quantile(s, 0.99) / np.median(s))

if os.path.exists(TUNING_RESULTS_PATH) and not FORCE_RETRAIN_SEARCH:
    tuning_df = pd.read_csv(TUNING_RESULTS_PATH)
    print(f"Loaded cached hyperparameter search results from {TUNING_RESULTS_PATH}")
else:
    tuning_results = []
    for (name, encoder_units), lr in itertools.product(ARCHITECTURES.items(), LEARNING_RATES):
        val_losses, sep_ratios, best_epochs = [], [], []
        for seed in SEEDS:
            tf.keras.backend.clear_session()
            np.random.seed(seed); tf.random.set_seed(seed)

            model = build_autoencoder(input_dim, encoder_units, learning_rate=lr)
            es = EarlyStopping(monitor="val_loss", patience=PATIENCE,
                               restore_best_weights=True)
            h = model.fit(X_train, X_train, validation_data=(X_val, X_val),
                          epochs=EPOCHS, batch_size=BATCH_SIZE, shuffle=True,
                          callbacks=[es], verbose=0)

            val_losses.append(float(np.min(h.history["val_loss"])))
            best_epochs.append(int(np.argmin(h.history["val_loss"]) + 1))
            sep_ratios.append(sep_ratio_from_model(model, X_norm))

        tuning_results.append({
            "model_name": name, "lr": lr,
            "architecture": arch_text_of(encoder_units),
            "mean_val_loss": float(np.mean(val_losses)),
            "std_val_loss": float(np.std(val_losses)),
            "mean_sep_ratio": float(np.mean(sep_ratios)),
            "mean_best_epoch": float(np.mean(best_epochs)),
            "num_params": model.count_params()
        })
        print(f"{name:<20} lr={lr:<7} mean_val_loss={np.mean(val_losses):.3e} "
              f"(+/-{np.std(val_losses):.1e})  sep(q99/med)={np.mean(sep_ratios):6.2f}")

    tuning_df = pd.DataFrame(tuning_results)
    tuning_df.to_csv(TUNING_RESULTS_PATH, index=False)
    print(f"Saved hyperparameter search results to {TUNING_RESULTS_PATH}")

display(tuning_df.sort_values("mean_val_loss").reset_index(drop=True))

In [ ]:
#Select purely by mean validation reconstruction loss (architecture + lr jointly).
#mean_sep_ratio is reported as a diagnostic, NOT used for selection.
best_row = tuning_df.sort_values("mean_val_loss").iloc[0]

best_model_name = best_row["model_name"]
best_encoder_units = ARCHITECTURES[best_model_name]
best_lr = float(best_row["lr"])

print("Selected architecture:", best_model_name, "=", best_row["architecture"])
print("Encoder units:", best_encoder_units, "| Learning rate:", best_lr)
print(f"Mean val loss: {best_row['mean_val_loss']:.3e} "
      f"(+/-{best_row['std_val_loss']:.1e} over {len(SEEDS)} seeds)")
print(f"Mean sep ratio (q99/med, reported only): {best_row['mean_sep_ratio']:.2f}")

In [ ]:
import pickle

MODEL_PATH = "sda_autoencoder.keras"
HISTORY_PATH = "sda_training_history.pkl"
FORCE_RETRAIN_MODEL = False  # set True to ignore the cache and retrain the final model

if os.path.exists(MODEL_PATH) and os.path.exists(HISTORY_PATH) and not FORCE_RETRAIN_MODEL:
    autoencoder = tf.keras.models.load_model(MODEL_PATH)
    autoencoder.summary()

    with open(HISTORY_PATH, "rb") as f:
        history_data = pickle.load(f)

    print(f"Loaded cached model from {MODEL_PATH}")
else:
    tf.keras.backend.clear_session()
    np.random.seed(42); tf.random.set_seed(42)

    autoencoder = build_autoencoder(input_dim, best_encoder_units,
                                    learning_rate=best_lr)
    autoencoder.summary()

    early_stop = EarlyStopping(monitor="val_loss", patience=PATIENCE,
                               restore_best_weights=True)

    history = autoencoder.fit(
        X_train, X_train,
        validation_data=(X_val, X_val),
        epochs=EPOCHS, batch_size=BATCH_SIZE, shuffle=True,
        callbacks=[early_stop], verbose=1
    )
    history_data = history.history

    autoencoder.save(MODEL_PATH)
    with open(HISTORY_PATH, "wb") as f:
        pickle.dump(history_data, f)

    print(f"Saved model to {MODEL_PATH} and history to {HISTORY_PATH}")

print("Final training loss:", history_data["loss"][-1])
print("Final validation loss:", history_data["val_loss"][-1])
print("Best epoch:", int(np.argmin(history_data["val_loss"]) + 1))
print("Best validation loss:", float(np.min(history_data["val_loss"])))

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_data["loss"], label="Training Loss")
plt.plot(history_data["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("SDA Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

# Anomaly Detection: Reconstruction Error &amp; Thresholding

The trained autoencoder learns to reconstruct *typical* 24-hour traffic profiles. Because anomalous profiles are rare and structurally different from the bulk of the data, the model should reconstruct them poorly. This gives a natural anomaly score: **per-profile reconstruction error**.

Methodology:
1. Compute the reconstruction error (RMSE across the 24 hours) for every profile in `X_train`, `X_val`, and the full dataset `X_norm`.
2. Use the **training-set** error distribution as the reference for "normal" behavior (the training set is not used elsewhere downstream, and early stopping already used validation loss, so this keeps the threshold derivation independent of the final scoring set).
3. Set a threshold at a high percentile of the training error distribution. Profiles across the full dataset whose error exceeds this threshold are flagged as anomalies.
4. Sanity-check the flags visually by comparing actual vs. reconstructed curves for the most anomalous and most typical profiles, then validate them quantitatively against the injected ground-truth labels (see the Evaluation section below).

Note: the original real-world deployment had no ground-truth anomaly labels, so this thresholding logic is written to stand on its own (visual/statistical justification, no label leakage). The synthetic version of this dataset happens to carry injected labels, which are used *only* for the evaluation section afterward — not for choosing the threshold itself.

In [ ]:
def reconstruction_error(X, X_hat):
    """Per-row RMSE between original and reconstructed profiles."""
    return np.sqrt(np.mean((X - X_hat) ** 2, axis=1))

recon_train = autoencoder.predict(X_train, verbose=0)
recon_val = autoencoder.predict(X_val, verbose=0)
recon_all = autoencoder.predict(X_norm, verbose=0)

err_train = reconstruction_error(X_train, recon_train)
err_val = reconstruction_error(X_val, recon_val)
err_all = reconstruction_error(X_norm, recon_all)

print(f"Train error - mean: {err_train.mean():.4f}, std: {err_train.std():.4f}, max: {err_train.max():.4f}")
print(f"Val error   - mean: {err_val.mean():.4f}, std: {err_val.std():.4f}, max: {err_val.max():.4f}")
print(f"All error   - mean: {err_all.mean():.4f}, std: {err_all.std():.4f}, max: {err_all.max():.4f}")

In [ ]:
#Sanity check: train and val error distributions should look similar,
#since both are drawn from the same (mostly-normal) population.
plt.figure(figsize=(9, 5))
plt.hist(err_train, bins=30, alpha=0.6, label="Train", density=True)
plt.hist(err_val, bins=30, alpha=0.6, label="Validation", density=True)
plt.xlabel("Reconstruction error (RMSE)")
plt.ylabel("Density")
plt.title("Train vs. Validation Reconstruction Error Distribution")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
#Threshold = a high percentile of the TRAINING error distribution.
#Rationale: X_train is assumed to be predominantly normal traffic, so its
#error distribution defines what "reconstructs well" looks like. 95th
#percentile means we expect ~5% of normal profiles to sit above threshold
#by chance; anything meaningfully above that rate in the full dataset is
#evidence of real anomalies rather than just distribution tail noise.
THRESHOLD_PERCENTILE = 95
threshold = np.percentile(err_train, THRESHOLD_PERCENTILE)

print(f"Reconstruction-error threshold ({THRESHOLD_PERCENTILE}th percentile of train error): {threshold:.4f}")
print(f"Sites in train exceeding threshold: {np.sum(err_train > threshold)} / {len(err_train)}")
print(f"Sites in val exceeding threshold:   {np.sum(err_val > threshold)} / {len(err_val)}")

In [ ]:
anomaly_flag = err_all > threshold

results_df = pd.DataFrame({
    "site": site_ids.values,
    "reconstruction_error": err_all,
    "is_anomaly": anomaly_flag
}).sort_values("reconstruction_error", ascending=False).reset_index(drop=True)

n_anomalies = int(anomaly_flag.sum())
print(f"Flagged {n_anomalies} anomalous profiles out of {len(anomaly_flag)} ({n_anomalies / len(anomaly_flag):.1%})")

#Break down flagged counts by subset, since only the synthetic rows have a
#known ground-truth answer (see the Quantitative Evaluation section below).
is_synthetic = site_ids.isin(ground_truth["site"]).values
print(f"  Synthetic subset: {int(anomaly_flag[is_synthetic].sum())} / {int(is_synthetic.sum())} flagged")
print(f"  Real subset:      {int(anomaly_flag[~is_synthetic].sum())} / {int((~is_synthetic).sum())} flagged")

display(results_df.head(10))

In [ ]:
sorted_err = np.sort(err_all)[::-1]

plt.figure(figsize=(9, 5))
plt.plot(sorted_err, marker=".", linestyle="none", markersize=4)
plt.axhline(threshold, color="red", linestyle="--",
           label=f"Threshold ({THRESHOLD_PERCENTILE}th pct of train err) = {threshold:.3f}")
plt.xlabel("Site rank (sorted by error, descending)")
plt.ylabel("Reconstruction error (RMSE)")
plt.title("Sorted Reconstruction Error Across All Sites")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(9, 5))
plt.hist(err_all, bins=40, alpha=0.7, label="All sites")
plt.axvline(threshold, color="red", linestyle="--", label=f"Threshold = {threshold:.3f}")
plt.xlabel("Reconstruction error (RMSE)")
plt.ylabel("Count")
plt.title("Distribution of Reconstruction Error (Full Dataset)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Visual Validation: Actual vs. Reconstructed Profiles

With no ground-truth labels, the strongest evidence that the flagged sites are genuinely anomalous is visual: the reconstruction should clearly fail to track the actual shape for high-error sites, and should track it closely for low-error (typical) sites.

In [ ]:
def plot_profile_reconstructions(indices, X, X_hat, site_ids, errors, title):
    n = len(indices)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4), sharey=True)
    if n == 1:
        axes = [axes]
    for ax, idx in zip(axes, indices):
        ax.plot(hour_cols, X[idx], marker="o", label="Actual", color="tab:blue")
        ax.plot(hour_cols, X_hat[idx], marker="x", linestyle="--",
               label="Reconstructed", color="tab:orange")
        ax.set_title(f"{site_ids.iloc[idx]}\nerror={errors[idx]:.3f}")
        ax.set_xlabel("Hour")
        ax.grid(True, alpha=0.3)
    axes[0].set_ylabel("Normalized traffic")
    axes[0].legend()
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

N_EXAMPLES = 5
top_anomaly_idx = np.argsort(err_all)[::-1][:N_EXAMPLES]
top_normal_idx = np.argsort(err_all)[:N_EXAMPLES]

plot_profile_reconstructions(top_anomaly_idx, X_norm, recon_all, site_ids, err_all,
                             "Most Anomalous Profiles: Actual vs. Reconstructed")
plot_profile_reconstructions(top_normal_idx, X_norm, recon_all, site_ids, err_all,
                             "Most Typical Profiles: Actual vs. Reconstructed")

## Quantitative Evaluation Against Ground Truth

This dataset's synthetic anomalies were deliberately injected with known labels — something impossible to have on the merged-in real profiles. That makes it possible to validate detection quality quantitatively on the **synthetic subset only**; the merge below scopes the evaluation to sites present in `ground_truth`, which excludes the real (re-identified) rows.

In [ ]:
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, roc_curve)

eval_df = results_df.merge(ground_truth, on="site", suffixes=("_pred", "_true"))

y_true = eval_df["is_anomaly_true"].values
y_pred = eval_df["is_anomaly_pred"].values
y_score = eval_df["reconstruction_error"].values

precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_score)

print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1 score:  {f1:.3f}")
print(f"ROC-AUC:   {auc:.3f}")

cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm, index=["Actual: normal", "Actual: anomaly"],
                     columns=["Predicted: normal", "Predicted: anomaly"])
display(cm_df)

print("\nRecall by injected anomaly type:")
display(eval_df[eval_df["is_anomaly_true"] == 1]
       .groupby("anomaly_type")["is_anomaly_pred"].mean().rename("recall"))

In [ ]:
fpr, tpr, _ = roc_curve(y_true, y_score)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve: Reconstruction Error as Anomaly Score")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
#Export flagged anomalies for downstream review / dashboarding.
OUTPUT_PATH = "anomalous_sites.csv"
results_df[results_df["is_anomaly"]].to_csv(OUTPUT_PATH, index=False)
print(f"Saved {int(anomaly_flag.sum())} flagged sites to {OUTPUT_PATH}")

In [ ]:
print(f"Summary: {n_anomalies} of {len(anomaly_flag)} sites flagged as anomalous "
     f"({n_anomalies / len(anomaly_flag):.1%}), using a threshold of {threshold:.4f} "
     f"({THRESHOLD_PERCENTILE}th percentile of training reconstruction error).")

## Conclusions &amp; Limitations

- A dense autoencoder, selected via a 6-architecture × 2-learning-rate × 3-seed grid search on mean-normalized 24-hour traffic profiles, was used to score every station (synthetic + re-identified real, merged) by reconstruction error.
- Thresholding at the 95th percentile of the training-set error distribution flags a station as anomalous; the flagged counts (overall and split by synthetic vs. real subset) are printed above.
- Ground-truth labels only exist for the synthetic subset, so precision/recall/F1/ROC-AUC (Quantitative Evaluation section) are computed there only. The real subset's flagged stations have no known answer to check against — they're visible in `results_df` but not in the evaluation metrics.
- Visual inspection (actual vs. reconstructed curves) corroborates the same conclusion qualitatively across both subsets.

**Limitations**
- The injected anomaly types (outage, spike, phase shift, inversion) are a designer's guess at what real anomalies look like — real-world anomalies may take different or subtler forms, so the precision/recall numbers should be read as validating the *methodology*, not guaranteeing it will catch every real anomaly type.
- The real subset's flagged stations are unverified — there's no ground truth to confirm they're genuinely anomalous rather than just unusual-but-normal for that station.
- The threshold is a single global percentile; it doesn't account for stations whose normal behavior is inherently more variable than others.
- Each day is scored independently; there's no use of history, so a station that is *consistently* unusual every day looks identical to a one-off event.

**Next steps**
- Benchmark against a simpler baseline (e.g., PCA reconstruction error or per-hour z-scores) to quantify what the autoencoder specifically adds over the injected-anomaly precision/recall numbers already measured here.
- If historical/multi-day data becomes available, extend to sequence models (e.g., LSTM autoencoder) to catch anomalies that unfold over time rather than within a single day.
- Cross-check the flagged real-subset stations against actual incident/outage records, if any exist, to get a real (rather than synthetic) precision estimate.